# NPTEL Study Notes: Pre-training Strategies in Large Language Models

## 1. Introduction: The Need for Contextual Embeddings

Traditional word embedding models like **Word2Vec** and **GloVe** produce *static embeddings*. In these models, a word always has the exact same vector representation, regardless of its context (e.g., the word "bat" in "a bat flew" vs. "hit with a bat" gets the same vector).

Modern pre-training strategies rely on the principle that **the complete meaning of a word is always contextual**. We need *dynamic/contextualized embeddings* where the vector representation of a word changes based on its surrounding sequence.

---

## 2. ELMo: Embeddings from Language Models (2018)

Before Transformers dominated, ELMo introduced a highly successful way to generate contextualized embeddings using **Recurrent Neural Networks (RNNs/LSTMs)**.

### 2.1 Core Architecture

* **Non-Transformer based:** ELMo relies on deep (stacked) bidirectional RNNs.
* **Bidirectional Scanning:** Unidirectional scanning (left-to-right) is insufficient for context. ELMo uses:
* **Forward LM:** Predicts the next word from left to right.
* **Backward LM:** Predicts the previous word from right to left.


* **Tokenization:** ELMo uses **character-level CNNs** (e.g., 2048 character-level filters) instead of word-level tokenization. This natively handles the Out-Of-Vocabulary (OOV / `[UNK]`) token problem.

### 2.2 Two-Stage Training Process

**Stage 1: Pre-training (Language Model Objective)**

* A stacked bi-directional RNN is trained to predict the next/previous word on massive text corporas.
* The hidden states ($h_t$) of this network essentially capture the word embeddings in context.

**Stage 2: Fine-Tuning (Task-Specific Application)**

1. **Freeze the LM:** The weights of the pre-trained RNN are frozen.
2. **Extract Hidden States:** For a specific task, pass the input sequence through the frozen LM.
3. **Combine Layers:** Instead of just using the top layer, ELMo combines hidden states from *multiple layers* (e.g., layer 1 and layer 2) using learned, task-specific weights.
4. **Add Task-Specific RNN/Layer:** Feed this combined embedding into a new, trainable RNN/linear layer to output task predictions (e.g., sentiment analysis).

### 2.3 Mathematical Representation of ELMo Embeddings

For a given token $k$, the final ELMo embedding is a weighted sum of its hidden states across all layers $j$ (from $j=0$ to $N$):

$$ELMo_k = \gamma \sum_{j=0}^{N} s_j h_{k, j}$$

* $h_{k, j}$: The concatenated hidden state (Forward + Backward) for token $k$ at layer $j$.
* $s_j$: Softmax-normalized, task-specific weights (determines which layer is most important for the specific task).
* $\gamma$: A task-specific scalar parameter that scales the entire ELMo vector.

*Note: During inference, the hidden states are calculated dynamically based on the current context, ensuring that "bat" gets different vectors depending on the surrounding words.*

---

## 3. Paradigm Shift: Pre-trained Language Models

With ELMo, only the *inputs* (embeddings) were pre-trained, and the task-specific architecture was trained from scratch. The paradigm shifted to **pre-training the entire model** (all parameters) and only slightly *fine-tuning* it for downstream tasks.

Depending on the Transformer blocks used, pre-trained models fall into three categories:

1. **Decoder-Only:** (e.g., GPT series) Uses masked self-attention (predicts next word; cannot look ahead).
2. **Encoder-Only:** (e.g., BERT, ALBERT, RoBERTa) Uses unmasked self-attention (can look bidirectionally).
3. **Encoder-Decoder:** (e.g., T5, BART) The standard original Transformer architecture.

---

## 4. BERT: Bidirectional Encoder Representations from Transformers

BERT (by Google) utilizes the **Encoder-only** portion of the Transformer. Because it lacks a decoder, its self-attention is **unmasked**, allowing every token to attend to every other token simultaneously (true bidirectionality).

### 4.1 Input Representation

BERT expects a specific input format to handle single sentences or sentence pairs:

* **`[CLS]` (Classification):** A special token prepended to the beginning of every sequence.
* *Why use `[CLS]`?* It attends to all other tokens but is not part of the actual sentence. Therefore, it provides an **unbiased aggregate representation** of the entire sequence, making it ideal for placing a classification head on top of it.


* **`[SEP]` (Separator):** A special token used to separate two different sentences (e.g., Question and Passage).
* **Embeddings:** The final input embedding fed to the BERT encoder is the sum of:
1. *Token Encoding* (the actual word piece)
2. *Segment Encoding* (indicates if the token belongs to Sentence A or Sentence B)
3. *Position Encoding* (indicates sequence order)



### 4.2 Pre-training Objectives (Self-Supervised)

Because BERT is bidirectional, it cannot use standard left-to-right next-word prediction. It relies on two simultaneous tasks:

#### Task 1: Masked Language Model (MLM)

We randomly mask **15%** of the input tokens and ask the model to predict them.

* *Why exactly 15%?* Masking too much (e.g., 50%) destroys the necessary context required to guess the words. Masking too little makes training inefficient.
* **The 80-10-10 Rule:** Of the 15% tokens chosen for masking:
* **80% of the time:** Replaced with the `[MASK]` token.
* **10% of the time:** Replaced with a *random token* (e.g., replacing "park" with "store").
* **10% of the time:** Kept exactly the same (unchanged).


* *Why the 10-10 variations?* If the model only ever predicts words when it sees `[MASK]`, it will fail to build strong representations for unmasked tokens. The random corruption and unchanged tokens force the model to thoroughly verify every word against its context.

#### Task 2: Next Sentence Prediction (NSP)

The model receives two sentences (W1 and W2) and must predict if W2 logically follows W1 in the original text.

* **Positive samples (50%):** Sentences appear consecutively in the corpus.
* **Negative samples (50%):** Sentences are paired randomly.
* Loss is calculated by placing a binary classification head on top of the `[CLS]` token.

### 4.3 Model Variants and Data

* **BERT-Base:** 12 Encoder layers.
* **BERT-Large:** 24 Encoder layers, larger hidden states, and 16 attention heads.
* **Training Data:** 2.5 Billion word Wikipedia corpus + 800 Million word Book corpus.

---

## 5. Fine-Tuning BERT: The Question Answering (QA) Example

Consider an extractive QA task (e.g., the SQuAD dataset). We have a Question ($Q$) and a Passage ($P$), and the answer is a specific span (start to end) within $P$.

**Input format:** `[CLS] Question [SEP] Passage [SEP]`

**Mechanism:**

1. Pass the sequence through BERT. Let the passage tokens output embeddings $T_1, T_2, ... T_N$.
2. Instead of using `[CLS]`, we place two classification layers (vectors) on top of *every token* in the passage:
* $C_b$ (Probability of being the Beginning/Start of the answer).
* $C_e$ (Probability of being the End of the answer).


3. For a given token $i$, the model outputs probability $p_b(i)$ and $p_e(i)$.

**Objective:** During inference, to find the exact answer span, we search across all possible combinations of start token $i$ and end token $j$, and select the span that maximizes:


$$\max_{i, j} (p_b(i) \times p_e(j))$$


**Subject to:** $j \ge i$ (The end of the answer span cannot occur before the beginning).

---

## 6. Post-BERT Evolution

* **RoBERTa:** Proved that the original BERT was significantly *undertrained*. By training longer on more data and removing the NSP objective, performance increased dramatically.
* **DistilBERT / TinyBERT:** Efforts focused on compressing the large BERT models into smaller, faster versions suitable for deployment, while retaining most of the performance.

---

## 7. Practice Questions for NPTEL Exams

**Q1. In the Masked Language Modeling (MLM) objective of BERT, 15% of the tokens are chosen for prediction. Out of this 15%, what percentage is replaced with a completely random token?**
a) 15%
b) 100%
c) 10%
d) 80%
**Answer: c) 10%**
*Explanation: Out of the chosen 15% tokens, 80% are replaced with `[MASK]`, 10% with a random word, and 10% are left unchanged.*

**Q2. Which of the following best describes the ELMo architecture?**
a) Deep Bidirectional Transformer Encoders
b) Forward-only LSTM networks
c) Deep Bidirectional LSTMs/RNNs
d) Decoder-only Transformers
**Answer: c) Deep Bidirectional LSTMs/RNNs**
*Explanation: ELMo relies on RNNs (specifically LSTMs) to process sequences, predicting next and previous words separately and concatenating their hidden states.*

**Q3. When fine-tuning BERT for an extractive Question Answering task, how is the final answer span selected?**
a) By feeding the `[CLS]` token representation into a generative decoder.
b) By maximizing the sum of the probabilities $p_b(i) + p_e(j)$.
c) By maximizing $p_b(i) \times p_e(j)$ subject to the constraint $j \ge i$.
d) By finding the token with the highest segment encoding score.
**Answer: c) By maximizing $p_b(i) \times p_e(j)$ subject to the constraint $j \ge i$.**
*Explanation: The start token $i$ and end token $j$ must maximize the joint probability, and logically, the end token $j$ must come after or at the same position as the start token $i$.*

**Q4. Why does BERT use the `[CLS]` token for sequence-level classification tasks instead of concatenating all token embeddings?**
a) `[CLS]` uses fewer attention heads.
b) `[CLS]` prevents gradient vanishing during backpropagation.
c) Concatenation is mathematically impossible in Transformers.
d) `[CLS]` is not part of the input sentence, ensuring its final representation is an unbiased aggregate of the entire sequence.
**Answer: d)**
*Explanation: Regular tokens are heavily biased towards their own meaning. `[CLS]` has no inherent meaning, so its self-attention cleanly aggregates the overall context. It also heavily reduces the parameter size of the classification head.*

---

## 8. Revision Summary

* **Static vs. Contextual:** Word2Vec/GloVe give one vector per word; ELMo/BERT give context-dependent vectors.
* **ELMo:** Bidirectional RNNs. Embeddings are linear combinations of internal hidden states from multiple layers (using task-specific parameters $\gamma$ and $s_j$). Uses character-level CNN tokenization.
* **Transformer Types:** Decoder-only (GPT), Encoder-only (BERT), Encoder-Decoder (T5).
* **BERT Architecture:** Encoder-only, true unmasked bidirectionality.
* **BERT Pre-training:**
1. **MLM:** Mask 15% of tokens (80% `[MASK]`, 10% random, 10% unchanged) to force contextual understanding.
2. **NSP:** Predict if Sentence B logically follows Sentence A to understand sentence relationships.


* **BERT Inputs:** Token + Position + Segment encodings. Bounded by `[CLS]` and `[SEP]`.
* **BERT Fine-Tuning QA:** Predict Start ($p_b$) and End ($p_e$) probabilities for every token. Extract maximum valid span ($p_b \times p_e$, where $j \ge i$).